<a href="https://colab.research.google.com/github/EtzionR/LM4GeoAI/blob/main/Solutions/SOL_3_Geo_LLM_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solution for Ex 3: **Basic Geo Agents and RAG**

### created by Etzion Harari | Geo-AI Course

[**https://github.com/EtzionR/LM4GeoAI**](https://github.com/EtzionR/LM4GeoAI)

## Prerequisites: pip install wikipedia, cohere & osmnx

In [1]:
print('Install wikipedia, cohere & osmnx using pip...\n')

!pip install -q wikipedia cohere osmnx

print('\nDone!')

Install wikipedia, cohere & osmnx using pip...

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.0/357.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 44.9 MB/s eta 0:00:00

Done!


## Imports

In [2]:
from difflib import get_close_matches
from transformers import pipeline
from getpass import getpass
from tqdm import tqdm

import pandas as pd
import numpy as np
import osmnx as ox
import wikipedia
import folium
import cohere

## Define cohere model

You can find documentation of cohere models here:

 [https://docs.cohere.com/docs/models#command](https://docs.cohere.com/docs/models#command)



In [3]:
MODEL  = "command-a-03-2025"
TEMP = .01

## Clone git Repo
[https://github.com/EtzionR/LM4GeoAI](https://github.com/EtzionR/LM4GeoAI)

In [4]:
%%bash
rm -rf LM4GeoAI
git clone https://github.com/EtzionR/LM4GeoAI.git

Cloning into 'LM4GeoAI'...


## Enter cohere API key

You can get your cohere API Key here (you should log-in / sign-in to cohere):

[https://dashboard.cohere.com/api-keys](https://dashboard.cohere.com/api-keys)

In [5]:
KEY = getpass("Please Enter COHERE KEY:\n")

len(KEY)

Please Enter COHERE KEY:
··········


40

## Init cohere Client

In [6]:
client = cohere.Client(api_key=KEY)
client

## Example of calling to cohere model

In [7]:
QUERY = 'What is panda bear?' # an example of a simple query

output = client.chat(model=MODEL,
                     message=QUERY,
                     temperature=TEMP).text

print(output)

A panda bear, often simply referred to as a panda, is a species of bear native to central China. Scientifically known as *Ailuropoda melanoleuca*, the panda is one of the most recognizable and beloved animals in the world, primarily due to its distinctive black and white coat. Here are some key facts about panda bears:

1. **Physical Characteristics**:  
   - Pandas have a stocky body, round face, and black patches around their eyes, ears, and across their body.  
   - Adults typically weigh between 70 to 125 kilograms (150 to 275 pounds) and stand about 75 to 80 centimeters (30 inches) tall at the shoulder.  

2. **Diet**:  
   - Pandas are primarily herbivores, with bamboo making up about 99% of their diet. They may also eat other vegetation, small animals, and fish occasionally.  
   - An adult panda can eat up to 12–38 kilograms (26–84 pounds) of bamboo daily.  

3. **Habitat**:  
   - Pandas are found in the mountainous regions of central China, particularly in Sichuan, Shaanxi, a

# --------------------------------------------------------------------

## Q1

#### 1) Implement a function that, given a location address, retrieves all points of interest (POIs) of specified place types within a defined radius from OpenStreetMap (OSM) using OSMnx.

#### 2) Based on the function defined in the previous section, display the output of the function for the following input parameters:

``` sh
location_adress = 'מגדלי עזריאלי'
place_type      = 'fuel'
radius          = 500
```

In [8]:
def get_pois_near_adress(location_adress, place_type, radius=500):
    """
    Load OSM POIs from given place_type

    :location_adress: location adress for searching OSM
    :place_type: type of point of interest (POI)
    :radius: radius around location_adress (in meters, int, default: 500)

    :return: selected dataframe
    """

    # Load data from OSM
    out = ox.features_from_address(location_adress, {"amenity": True}, radius)
    out = out[['name','amenity','geometry']]
    out = out.rename(columns={'amenity':'place_type'})

    # Find the closets place_type (using SequenceMatcher)
    place_type = get_close_matches(place_type,
                                   [*out.place_type.unique()],
                                   n=1,
                                   cutoff=.0)[0]

    # select the place_type
    dataframe = out[out.place_type==place_type]

    return dataframe

# ------------------------------------------------------------------------------


# get_place_info application:

location_adress = 'מגדלי עזריאלי'
place_type      = 'fuel'

get_pois_near_adress(location_adress, place_type)

name place_type                   geometry
element id                                                  
node    442893138  דלק       fuel  POINT (34.79586 32.07468)
        442893144  דלק       fuel  POINT (34.79449 32.07144)
        442898671   פז       fuel   POINT (34.79724 32.0719)

## Q2

#### 1) Implement a function that computes the driving route between two given addresses using the OSM road network.

#### 2) Based on the function defined in the previous section, display the output for the following input parameters:

``` sh
adress_1 = "אוניברסיטת תל אביב"
adress_2 = 'מגדלור רידינג'
```

In [9]:
def shortest_path_between_adresses(adress_1, adress_2,  gap=.25):
    """
    Calculate shortest path between two adresses using OSM roads network

    :adress_1: origin adress (source)
    :adress_2: destination adress (target)
    :gap: buffer ratio from bbox (float, default: .25)

    :return: shortest path between adress_1 to adress_2 (coordinates)
    """

    # gecode to both adresses
    y1, x1 = ox.geocode(adress_1)
    y2, x2 = ox.geocode(adress_2)

    # build the bbox
    delta = max(abs(x1-x2)*gap, abs(y1-y2)*gap)
    bbox = [min(x1,x2)-delta, min(y1,y2)-delta, max(x1,x2)+delta, max(y1,y2)+delta]

    # get the roads network graph
    G = ox.graph.graph_from_bbox(bbox, truncate_by_edge =True, network_type ='drive', simplify =False)

    # find the origin & destination nodes
    node1 = ox.distance.nearest_nodes(G, x1, y1)
    node2 = ox.distance.nearest_nodes(G, x2, y2)

    # calculate path between origin to destination
    route = ox.routing.shortest_path(G, orig=node1, dest=node2)
    route_xy = [(G.nodes[osmid]['x'], G.nodes[osmid]['y']) for osmid in route]

    return route_xy

# ------------------------------------------------------------------------------


# shortest_path_between_adresses application:

ad1 = "אוניברסיטת תל אביב"
ad2 = 'מגדלור רידינג'

path = shortest_path_between_adresses(ad1, ad2)

print(f'Number of nodes in the output path: {len(path)}\n\nFirst 5 points:\n{path[:5]}')

Number of nodes in the output path: 119

First 5 points:
[(34.8037004, 32.1107766), (34.8037061, 32.1107118), (34.8037047, 32.1105204), (34.8036907, 32.1103309), (34.8036844, 32.1101144)]


## Q3

#### 1) Write the system prompt for a Cohere client that defines how the LLM agent should use the functions from Q1 and Q2, in accordance with the Model Context Protocol (MCP).

#### 2) Define the tools schema that specifies how the LLM agent can invoke the functions from Q1 and Q2 for the Cohere client, following the Cohere MCP documentation.

#### 3) Define a dictionary named **tools_map** that maps the function names specified in Q1 and Q2 to their corresponding function implementations.

You may refer to the following documentation for guidance:
https://docs.cohere.com/page/basic-tool-use

In [10]:
# system prompt
system_prompt = """You are an AI assistant with access to the following tools:
get_pois_near_adress tool, which enable you to run osm query.
shortest_path_between_adresses tool, which enable you to calculate route between to given place names."""

# tools schema
tools = [
    {
        "name": "get_pois_near_adress",
        "description": "can return every type of geographic location near some given adress",
        "parameter_definitions": {
            "location_adress": {
                "description": "free text location adress",
                "type": "str",
                "required": True
            },
            "place_type": {
                "description": "type of point of interest (POI)",
                "type": "str",
                "required": True
            },
            "radius": {
                "description": "radius of search in meters",
                "type": "float",
                "required": False
            }
        }
    },

    {
        "name": "shortest_path_between_adresses",
        "description": "Calculate shortest path between two adresses using OSM roads network",
        "parameter_definitions": {
            "adress_1": {
                "description": "adress of the origin point (free text)",
                "type": "str",
                "required": True
            },
            "adress_2": {
                "description": "adress of the destination point (free text)",
                "type": "str",
                "required": True
            },
            "gap": {
                "description": "buffer ratio of the bbox to load roads network",
                "type": "float",
                "required": False
            }
        }
    },
]

# tools map
tools_map = {get_pois_near_adress.__name__: get_pois_near_adress,
                 shortest_path_between_adresses.__name__: shortest_path_between_adresses}
tools_map

{'get_pois_near_adress': <function __main__.get_pois_near_adress(location_adress, place_type, radius=500)>,
 'shortest_path_between_adresses': <function __main__.shortest_path_between_adresses(adress_1, adress_2, gap=0.25)>}

## Q4

#### Using the Cohere client, invoke the LLM with the following input:

``` sh
user_input = "I want you to return every cafe near Azrieli Center"
```

#### The cohere client should utilize the function defined in Q1 to fulfill the request.

In [11]:
user_input = f"I want you to return every cafe near Azrieli Center"


response = client.chat(
    model = MODEL,
    message = user_input,
    preamble = system_prompt,
    tools = tools
)

print(f"Model output: {response.text}\n\n")

out = tools_map[response.tool_calls[0].name](**response.tool_calls[0].parameters)
out

Model output: I will use the get_pois_near_adress tool to find every cafe near Azrieli Center.




name place_type                   geometry
element id                                                            
node    553022802       קפה שרגא       cafe    POINT (34.7903 32.0707)
        1207157419           Uno       cafe  POINT (34.78908 32.07748)
        3465156312        רולדין       cafe  POINT (34.78693 32.07173)
        6469411087        לנדוור       cafe  POINT (34.79164 32.07452)
        6469411088       קפה קפה       cafe   POINT (34.79172 32.0747)
        6644165508         Aroma       cafe  POINT (34.78769 32.07308)
        7211406506   Max Brenner       cafe   POINT (34.78677 32.0715)
        10784964980      Arcaffe       cafe  POINT (34.78864 32.07829)
        10812416453       BEITEA       cafe  POINT (34.78798 32.07223)
        11802167414      Arcaffe       cafe  POINT (34.79304 32.07731)
        12032015683          mae       cafe  POINT (34.78829 32.07195)
        12139988888     קפה עוטף       cafe  POINT (34.78739 32.07308)
        12635329119       רולדין       cafe   POINT (34.79232 32.0743)
        13338979218         Biga       cafe  POINT (34.78691 32.07308)
        13472351447      Arcaffe       cafe  POINT (34.79172 32.07467)
        13801746801          NaN       cafe   POINT (34.7905 32.07783)

## Q5

#### Visualize the outputs from Q4 on a Folium map.

In [12]:
fmap = folium.Map(location=[out.geometry.y.median(), out.geometry.x.median()],
                  zoom_start=15.5, width=500, height=500)

folium.GeoJson(out).add_to(fmap)

fmap

## Q6

#### Using the Cohere client, invoke the LLM with the following input:

``` sh
user_input = f"אני רוצה לדעת מה המסלול מחוף מציצים למגדלי עזריאלי"
```

#### The cohere client should utilize the function defined in Q2 to fulfill the request.

In [13]:
user_input = f"אני רוצה לדעת מה המסלול מחוף מציצים למגדלי עזריאלי"


response = client.chat(
    model = MODEL,
    message = user_input,
    preamble = system_prompt,
    tools = tools
)

print(f"Model output: {response.text}\n\n")

out = tools_map[response.tool_calls[0].name](**response.tool_calls[0].parameters)

print(f'Output length: {len(out)}')

Model output: אשתמש בכלי shortest_path_between_adresses כדי למצוא את המסלול הקצר ביותר מחוף מציצים למגדלי עזריאלי.


Output length: 138


## Q7

#### Visualize the outputs from Q6 on a Folium map.

In [14]:


fmap = folium.Map(location=[np.array(out)[:,1].mean(), np.array(out)[:,0].mean()],
                  zoom_start=14, width=500, height=500)

folium.PolyLine(
    locations=[(y,x) for x,y in out],
    color="blue",
    weight=5,
    tooltip=user_input).add_to(fmap)

folium.Marker([out[0][-1] ,out[0][0]],
                  tooltip='Start',
                  zIndexOffset=1,
                  icon=folium.Icon(color='blue', icon='play')).add_to(fmap)

folium.Marker([out[-1][-1] ,out[-1][0]],
                  tooltip='End',
                  zIndexOffset=1,
                  icon=folium.Icon(color='blue', icon='pause')).add_to(fmap)

fmap

## Q8

#### 1) Develop a Retrieval-Augmented Generation (RAG) pipeline to describe the environment surrounding a given geographic coordinate. The pipeline should perform the following steps:

- Accept X and Y coordinates in WGS84 DD (EPSG:4326).

- Retrieve OSM **buildings** and **POIs** within a configurable distance radius (optionally, retrieve a single entity).

- Extract and return the name of the retrieved entity from OSM.

- Identify the corresponding **Wikipedia** page for the retrieved entity.

- Retrieve the **first 256 characters** of content from the Wikipedia page.

- Use the retrieved context to enhance the LLM’s output, enabling it to generate a detailed description of the environment at the specified coordinates based on Wikipedia knowledge.

- Return the LLM-generated description.

#### 2) Apply the RAG pipeline using the following input coordinates:

``` sh
X = -77.0352332
Y = 38.8893845
```



In [17]:

def get_pois_from_coordinates(x,y, poi_search_radius=100):
    """
    Load all OSM POIs in a given radius around given point

    :x: x coordiante (WGS84 GEO DD EPSG:4326, float)
    :y: y coordiante (WGS84 GEO DD EPSG:4326, float)
    :dist: distance around the point in meters (float | int)

    return: all place name in the given radius
    """

    point = (y,x)

    options = ox.features.features_from_point(center_point = point, dist=poi_search_radius, tags = {'building': True, "amenity": True})
    options = options[['name']].dropna().drop_duplicates()

    return [*options.name]

def get_context_from_wikipedia(page_name, context_length=256):
    """
    Load the content of wikipedia page by given page name

    :page_name: given wikipedia page name (str)
    :context_length: maximum context length (int)

    return: context knowledge on the page name
    """

    page = wikipedia.page(wikipedia.search(page_name)[0])

    return page.content[:context_length]

def describe_location(x,y, poi_search_radius=100):
    """
    describe a location given its coordinates

    :x: x coordiante (WGS84 GEO DD EPSG:4326, float)
    :y: y coordiante (WGS84 GEO DD EPSG:4326, float)
    :dist: distance around the point in meters (float | int)

    return: cohere model description on the location
    """

    place_names = get_pois_from_coordinates(x,y, poi_search_radius=poi_search_radius)
    context = get_context_from_wikipedia(place_names[0])

    response = client.chat(model = MODEL,
                           message = f'Please describe {place_names[0]} given the context knowledge from wikipedia:\n\n{context}')
    return response.text

output = describe_location(-77.0352332, 38.8893845)

print(output)

The **Washington Monument** is an iconic 555-foot (169 m) tall marble, granite, and sandstone obelisk located on the **National Mall** in **Washington, D.C.**, United States. It was constructed to honor **George Washington**, a **Founding Father** and the first President of the United States, in recognition of his leadership during the American Revolution and his role in the nation's early government. The monument stands prominently on the National Mall, positioned east of the **Reflecting Pool**, and is a central feature of the area's landscape, visible from many parts of the city.

Construction of the monument began in **1848**, but it was halted due to funding issues and the American Civil War, and was not completed until **1884**. It was officially opened to the public in **1888**. The obelisk's design, by architect **Robert Mills**, was inspired by ancient Egyptian precedents, symbolizing timelessness and grandeur. The exterior is composed of white marble and granite, with a small

## Q9

#### 1) Load the queries from the file from this path:
``` sh
queries_path = 'LM4GeoAI/Data/AgentQueries.csv'
```

#### 2) Split the queries table to train set (20%) and test set (80%)

#### 3) Use a language model to classify the **test** queries ("query" column) to the correct type ("query_type" column). You can use E5 model to perform this classification (few-shot classification, using the train set).

``` sh
MODEL = "intfloat/multilingual-e5-large"
```

#### 4) Report your results as Accuracy Score and display the Confusion Matrix

In [18]:
queries_path = 'LM4GeoAI/Data/AgentQueries.csv'

example_queries = pd.read_csv(queries_path)

example_queries.sample(5)

,Number,Query_Type,Query
85,86,describe location,"Give me context on this location: -1.2921, 36...."
28,29,shortest path,Calculate the fastest path from my office in F...
16,17,shortest path,What's the fastest way from Barcelona airport ...
119,120,get satellite image,Display satellite imagery of the Great Wall of...
135,136,get ground image,I'd like to see pictures taken at the Taj Maha...


In [19]:
P = .2

train_set = np.random.random(len(example_queries))<P
test_set  = train_set==False

train = example_queries[train_set]
test  = example_queries[test_set]

print(f'Size of Train set: {train.shape}\nSize of Test set:  {test.shape}\n\nTrain query types counts:\n')

train.Query_Type.value_counts()

Size of Train set: (26, 3)
Size of Test set:  (124, 3)

Train query types counts:



,count
Query_Type,
get satellite image,9
get ground image,6
shortest path,4
describe location,4
get pois near location,3


In [20]:
EMODEL = "intfloat/multilingual-e5-large"

embedder = pipeline("feature-extraction", model=EMODEL)
get_text_embedding = lambda text: np.array(embedder(text)[0]).max(0).reshape(-1)

embedding_matrix = np.array([get_text_embedding(query) for query in tqdm(example_queries.Query)])
embedding_matrix.shape

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

100%|██████████| 150/150 [01:11<00:00,  2.08it/s]


(150, 1024)

In [21]:
confusion_matrix = {('Test',typ):{('Pred',typ):0 for typ in example_queries.Query_Type.unique()}
                                                 for typ in example_queries.Query_Type.unique()}

query_type_train = [*example_queries[train_set].Query_Type]
query_type_test  = [*example_queries[test_set ].Query_Type]

matmul = embedding_matrix[train_set]@embedding_matrix[test_set].T

query_type_test_pred = [query_type_train[matmul[:,i].argmax()] for i in range(test_set.sum())]

acc = np.mean([yi==pi for yi,pi in zip(query_type_test, query_type_test_pred)])
print(f'\nFew-shot classification Accuracy Score is {round(100*acc,1)}%\n(Model: {EMODEL})\n\n')

for yi,pi in zip(query_type_test, query_type_test_pred):
    confusion_matrix[('Test',yi)][('Pred',pi)]+=1

confusion_matrix = pd.DataFrame(confusion_matrix).replace(0, '').T
confusion_matrix


Few-shot classification Accuracy Score is 82.3%
(Model: intfloat/multilingual-e5-large)




Pred                         \
                            shortest path get pois near location   
Test shortest path                     26                          
     get pois near location             6                     11   
     describe location                                             
     get satellite image                                           
     get ground image                                              

                                                                   \
                            describe location get satellite image   
Test shortest path                                                  
     get pois near location                 4                   4   
     describe location                     26                       
     get satellite image                                       20   
     get ground image                                           5   

                                              
                            get ground image  
Test shortest path                            
     get pois near location                2  
     describe location                        
     get satellite image                   1  
     get ground image                     19

## Q - Bonus

#### Design and implement a custom LLM agent or RAG (Retrieval-Augmented Generation) pipeline tailored to a specific task of your choice. You are encouraged to use your creativity and imagination to define:

- The objective of the agent or pipeline.

- The sources of information or knowledge retrieval mechanisms (e.g., databases, APIs, websites, or documents).

- The input format and how the agent will process it.

- Any additional processing or enrichment steps to improve the quality or relevance of the results.

- The expected output and how it addresses the defined task.

- Document your design choices, implementation approach, and demonstrate the pipeline with a concrete example input.

Create by Etzion Harari | Geo-AI Course | [https://github.com/EtzionR/LM4GeoAI](https://github.com/EtzionR/LM4GeoAI)